In [7]:
import re, csv, glob, torch
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForCausalLM

# Config
MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
FILES = ["recipe.txt","non_fiction.txt","hp.txt","cricket.txt","orogeny.txt"]
CORPUS_PATH = "corpus_concat.txt"
NEEDLE_CODE = "BaKaLuFFY22"
NEEDLE_SENTENCE = f"The secret ticket code is {NEEDLE_CODE}."
QUESTION = "What is the secret ticket code?\nRespond on one line only:\nFinal: <code>."
# ------------------------------------------------


# 1) Concatenating .txt files to build corpus with whitespace normalization
def clean_text(s: str) -> str:
    return re.sub(r"\s+", " ", s).strip()

texts = [clean_text(Path(p).read_text(encoding="utf-8", errors="ignore")) for p in FILES]
final_text = " ".join(texts)
Path(CORPUS_PATH).write_text(final_text, encoding="utf-8")
print("Concatenated corpus saved to:", CORPUS_PATH)

# 2) Loading tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True, use_fast=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL,
    dtype="auto",
    device_map="auto",
    low_cpu_mem_usage=True,
    attn_implementation="sdpa",
)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

# 3) Insert needle at 3 different positions and getting 3 different files for it
def build_with_needle(text: str, position: str):
    sep = "\n\n"
    insert = f"{sep}{NEEDLE_SENTENCE}{sep}"
    if position == "start":
        idx_char = 0
    elif position == "middle":
        idx_char = len(text) // 2
    elif position == "end":
        idx_char = len(text)
    else:
        raise ValueError("position must be start/middle/end")
    new_text = text[:idx_char] + insert + text[idx_char:]
    out_path = f"long_{position}.txt"
    Path(out_path).write_text(new_text, encoding="utf-8")
    # print(f"{position.title()}: inserted at char {idx_char} of {len(text)}")
    print(f"{position.title()}: needle inserted.")

    return out_path

# 4) Ensuring code uniqueness
corpus = Path(CORPUS_PATH).read_text(encoding="utf-8")
assert NEEDLE_CODE not in corpus, "Code already in corpus!"

tokens = tokenizer(corpus, add_special_tokens=False).input_ids
print("Number of tokens:", len(tokens))

# 5) Creating long contexts
made = [(pos, build_with_needle(corpus, pos)) for pos in ["start","middle","end"]]

# 6) Helper to find position
def needle_token_index(chat_str: str, needle_str: str):
    start_char = chat_str.find(needle_str)
    if start_char < 0:
        return -1, float("nan"), None
    enc = tokenizer(
        chat_str,
        return_tensors="pt",
        add_special_tokens=False,         # chat template already includes special tokens
        return_offsets_mapping=True,      # fast tokenizer gives char spans
    )
    offsets = enc["offset_mapping"][0].tolist()
    tok_idx = -1
    for i, (a, b) in enumerate(offsets):
        # skip special tokens (usually (0,0)); find the first token that covers the char
        if a == b:
            continue
        if a <= start_char < b:
            tok_idx = i
            break
    total = int(enc["input_ids"].shape[1])
    rel = (tok_idx / total) if tok_idx >= 0 else float("nan")

    return tok_idx, rel, enc

# 7) Run deterministically
csv_path = "results.csv"
with open(csv_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["version","position","tokens_total","needle_token_idx","needle_rel_pos","prediction","correct"])

    for position, fpath in made:
        long_text = Path(fpath).read_text(encoding="utf-8")
        messages = [
            {"role":"system","content":"You are a precise assistant. Reply with exactly one line in the format: 'Final: <code>'."},
            {"role":"user","content": long_text + "\n\n" + QUESTION},
        ]

        # Format as a chat string (don’t tokenize yet)
        chat_str = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

        # Find the needle index in token space (post-template)
        needle_idx, needle_rel, enc = needle_token_index(chat_str, NEEDLE_SENTENCE)
        total_tokens = int(enc["input_ids"].shape[1])
        print(f"[{position}] tokens_total={total_tokens} | needle_token_idx={needle_idx} | needle_rel={needle_rel:.6f}")

        # Build model inputs from our encoded dict (avoid Tensor.items() pitfall)
        inputs = {
            "input_ids": enc["input_ids"].to(model.device),
            "attention_mask": enc["attention_mask"].to(model.device),
        }

        with torch.no_grad():
            out = model.generate(
                **inputs,
                max_new_tokens=32,
                do_sample=False,
                num_beams=1,
                eos_token_id=tokenizer.eos_token_id,
                pad_token_id=tokenizer.eos_token_id,
            )
        gen = out[0, inputs["input_ids"].shape[1]:]
        generated = tokenizer.decode(gen, skip_special_tokens=True)

        m = re.search(r"Final:\s*([A-Za-z0-9\-]+)", generated)
        prediction = m.group(1) if m else ""
        correct = int(prediction == NEEDLE_CODE)

        writer.writerow(["v2", position, total_tokens,
                         int(needle_idx) if needle_idx >= 0 else "NA",
                         round(float(needle_rel), 6) if needle_idx >= 0 else "NA",
                         prediction, correct])
        print(f"[{position}] Prediction: {prediction or '∅'} | Correct: {correct} | needle_idx={needle_idx} rel={needle_rel:.6f}")

print("Results saved to", csv_path)


Concatenated corpus saved to: corpus_concat.txt
Number of tokens: 10335
Start: needle inserted.
Middle: needle inserted.
End: needle inserted.
[start] tokens_total=10400 | needle_token_idx=29 | needle_rel=0.002788
[start] Prediction: ∅ | Correct: 0 | needle_idx=29 rel=0.002788
[middle] tokens_total=10402 | needle_token_idx=5188 | needle_rel=0.498750
[middle] Prediction: ∅ | Correct: 0 | needle_idx=5188 rel=0.498750
[end] tokens_total=10400 | needle_token_idx=10364 | needle_rel=0.996538
[end] Prediction: ∅ | Correct: 0 | needle_idx=10364 rel=0.996538
Results saved to results.csv
